# *ASSIGNMENT-3: Sending messages through a network*

In [ ]:
import networkx as nx
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import random
import numpy as np
from itertools import combinations

In [ ]:
MESSAGE_TRUE = "Low"
MESSAGE_MUTATED = "Moderate"
MESSAGE_FAKE = "Fake"
MESSAGE_NONE = None

In [ ]:
BASE_DIR = Path().resolve().parent.parent
DATASET_PATH = BASE_DIR / "data" / "anime_filtered.csv"
df = pd.read_csv(DATASET_PATH)

In [ ]:
def build_graph(filtered):
    G = nx.Graph()
    for _, row in filtered.iterrows():
        G.add_node(row["mal_id"], title=row["title"])
    for (_, a), (_, b) in combinations(filtered.iterrows(), 2):
        genres_a = set(str(a["genres"]).split("|"))
        genres_b = set(str(b["genres"]).split("|"))
        if len(genres_a & genres_b) >= 2:
            G.add_edge(a["mal_id"], b["mal_id"])
    return G

## Verifiers and Unintentional nodes

In [ ]:
def assign_roles(G, n_verifiers=5, n_unintentional=5):
    roles = {node: "normal" for node in G.nodes()}
    
    verifiers = random.sample(G.nodes(), n_verifiers)
    for v in verifiers:
        roles[v] = "verifier"
    
    remaining = [n for n in G.nodes() if n not in verifiers]
    unint = random.sample(remaining, n_unintentional)
    for u in unint:
        roles[u] = "unintentional"
    
    return roles

In [ ]:
def adopt_message(node, G, messages, threshold):
    neighbors = list(G.neighbors(node))
    if not neighbors:
        return MESSAGE_NONE
    
    # conteggio dei messaggi dei vicini
    counts = {}
    for n in neighbors:
        msg = messages[n]
        if msg is not None:
            counts[msg] = counts.get(msg, 0) + 1
    
    if not counts:
        return MESSAGE_NONE
    
    # messaggio più frequente
    majority_msg = max(counts, key=counts.get)
    frac = counts[majority_msg] / len(neighbors)
    
    if frac >= threshold:
        return majority_msg
    else:
        return MESSAGE_NONE

In [ ]:
def simulate_spread(G, roles, threshold=0.3, max_steps=20):
    messages = {node: MESSAGE_NONE for node in G.nodes()}
    
    # Verifiers iniziano con il messaggio corretto
    for node, role in roles.items():
        if role == "verifier":
            messages[node] = MESSAGE_TRUE
    
    for step in range(max_steps):
        new_messages = messages.copy()
        
        for node in G.nodes():
            role = roles[node]
            
            if role == "verifier":
                continue
            
            if role == "unintentional":
                if messages[node] is None:
                    adopted = adopt_message(node, G, messages, threshold)
                    if adopted == MESSAGE_TRUE:
                        new_messages[node] = MESSAGE_MUTATED
                continue
            
            if role == "normal":
                adopted = adopt_message(node, G, messages, threshold)
                new_messages[node] = adopted
        
        if new_messages == messages:
            break
        
        messages = new_messages
    return messages


In [ ]:
def analyze(messages):
    total = len(messages)
    true_count = sum(1 for m in messages.values() if m == MESSAGE_TRUE)
    mutated_count = sum(1 for m in messages.values() if m == MESSAGE_MUTATED)
    
    return {
        "true": true_count / total,
        "mutated": mutated_count / total,
        "none": 1 - (true_count + mutated_count) / total
    }


## Adding Social Bots

In [ ]:
def assign_bots(G, roles, n_bots=3, mode="degree"):
    if mode == "degree":
        centrality = dict(G.degree())
    else:
        centrality = nx.betweenness_centrality(G)
    
    top_nodes = sorted(centrality, key=centrality.get, reverse=True)[:n_bots]
    
    for b in top_nodes:
        roles[b] = "bot"
    
    return roles


In [ ]:
def simulate_spread_with_bots(G, roles, threshold=0.3, max_steps=20):
    messages = {node: MESSAGE_NONE for node in G.nodes()}
    
    for node, role in roles.items():
        if role == "verifier":
            messages[node] = MESSAGE_TRUE
    
    for step in range(max_steps):
        new_messages = messages.copy()
        
        for node in G.nodes():
            role = roles[node]
            
            if role == "verifier":
                continue
            
            if role == "bot":
                neighbors = list(G.neighbors(node))
                if any(messages[n] is not None for n in neighbors):
                    new_messages[node] = MESSAGE_FAKE
                continue
            
            if role == "unintentional":
                if messages[node] is None:
                    adopted = adopt_message(node, G, messages, threshold)
                    if adopted == MESSAGE_TRUE:
                        new_messages[node] = MESSAGE_MUTATED
                continue
            
            if role == "normal":
                # se un vicino ha fake → adotta fake
                if any(messages[n] == MESSAGE_FAKE for n in G.neighbors(node)):
                    new_messages[node] = MESSAGE_FAKE
                    continue
                
                adopted = adopt_message(node, G, messages, threshold)
                new_messages[node] = adopted
        
        if new_messages == messages:
            break
        
        messages = new_messages
    
    return messages


In [ ]:
def experiment_unintentional(G, threshold=0.3, max_unintentional=30):
    results = []
    
    for u in range(max_unintentional + 1):
        roles = assign_roles(G, n_verifiers=5, n_unintentional=u)
        messages = simulate_spread(G, roles, threshold)
        stats = analyze(messages)
        results.append((u, stats["mutated"]))
    
    return results

In [ ]:
def plot_phase_transition(results):
    x = [r[0] for r in results]
    y = [r[1] for r in results]
    
    plt.figure(figsize=(8,6))
    plt.plot(x, y, marker='o')
    plt.xlabel("Numero di Unintentional nodes")
    plt.ylabel("Percentuale di messaggi MUTATED")
    plt.title("Phase transition — diffusione del messaggio mutato")
    plt.grid(True)
    plt.show()

In [ ]:
G = build_graph(df)
roles = assign_roles(G, n_verifiers=5, n_unintentional=10)
messages = simulate_spread(G, roles, threshold=0.3)
print(analyze(messages))

roles_bots = assign_bots(G, roles, n_bots=3, mode="betweenness")
messages_bots = simulate_spread_with_bots(G, roles_bots, threshold=0.3)
print(analyze(messages_bots))

results = experiment_unintentional(G, threshold=0.3, max_unintentional=40)
plot_phase_transition(results)